In [1]:
import os
import csv
import glob
from datetime import datetime

In [4]:
column_names = ["time", "sensor", "aX(g)", "aY(g)", "aZ(g)", "ωX(°/s)", "ωY(°/s)", "ωZ(°/s)", "θX(°)", "θY(°)", "θZ(°)"]
sensors = ['FD', '5D', 'B1', 'C8', 'D9']

def fix_time(time_str):
    time_part, milli_part = time_str.split('.')
    
    # 2. 毫秒部分補足三位數 (zfill 會在左邊補 0)
    # 例如 "4" -> "004", "40" -> "040"
    fixed_milli = milli_part.zfill(3)
    
    # 3. 拼回去
    return f"{time_part}.{fixed_milli}"


path_txt = r"C:\Users\User\Desktop\高三專題\數據\**\*.txt"
target_files_txt = glob.glob(path_txt, recursive=True)    

def txt_to_csv():
    for input_txt in target_files_txt:
        # 跳過已經是 csv 的檔案或正在執行的腳本
        output_csv = input_txt.replace('.txt', '.csv')
        
        # 抓 initial time
        with open(input_txt, 'r', encoding='utf-8') as f_in:
             
            next(f_in) 
            
            second_line = f_in.readline()
            parts = second_line.split() 
            txtinitial_time = parts[1]
            initial_time = datetime.strptime(fix_time(txtinitial_time), "%H:%M:%S.%f")
        

        with open(input_txt, 'r', encoding='utf-8') as f_in, \
             open(output_csv, 'w', newline='', encoding='utf-8') as f_out:
                
            writer = csv.writer(f_out)
        
            # 寫入標題
            writer.writerow(column_names)
        
            # 略過原始文字檔的第一列
            next(f_in) 
            
            for line in f_in:
                parts = line.split()
                
                current_time = datetime.strptime(fix_time(parts[1]), "%H:%M:%S.%f")
                time = (current_time - initial_time).total_seconds()
                sensor = parts[2].split(':')[-1].replace(')', '') 
                aX = parts[3]  # 加速度X
                aY = parts[4]  # 加速度Y
                aZ = parts[5]  # 加速度Z
                ωX = parts[6]  # 角速度X
                ωY = parts[7]  # 角速度Y
                ωZ = parts[8]  # 角速度Z
                θX = parts[9]  # 姿態角X
                θY = parts[10] # 姿態角Y
                θZ = parts[11] # 姿態角Z
                

                row_to_keep = [time, sensor, aX, aY, aZ, ωX, ωY, ωZ, θX, θY, θZ]
                
                # 寫入 CSV
                writer.writerow(row_to_keep) 


path_csv = r"C:\Users\User\Desktop\高三專題\數據\**\*.csv"
target_files_csv = glob.glob(path_csv, recursive=True)  

def split_sensors():
    for input_csv in target_files_csv:
        # 1. 取得大檔案所在的目錄 (例如: ...\straight\1)
        base_dir = os.path.dirname(input_csv)
        
        # 2. 取得大檔案的純檔名 (例如: s1)
        file_raw_name = os.path.basename(input_csv).replace('.csv', '')

        # 建立一個字典來存放「檔案管家」
        # 這樣你就有 5 個已經打開且寫好標題的 CSV 檔案
        files = {}
        writers = {}
        
        # 為每個 Sensor 準備好輸出的 CSV
        for s in sensors:
            sensor_dir = os.path.join(base_dir, s)
            out_path = os.path.join(sensor_dir, f"{file_raw_name}.{s}.csv")
            
            f_out = open(out_path, 'w', newline='', encoding='utf-8')
            writer = csv.writer(f_out)
            writer.writerow(column_names) # 先寫入標題
            files[s] = f_out
            writers[s] = writer

        # 3. 開始讀取大檔案並「一筆一筆丟」
        with open(input_csv, 'r', encoding='utf-8') as f_in:
            reader = csv.reader(f_in)
            next(reader) # 跳過標題
            
            for row in reader:
                # row[1] 就是 Sensor ID
                s_id = row[1]
                writers[s_id].writerow(row) # 依照 ID 丟進對應的檔案



In [3]:
txt_to_csv()

In [5]:
split_sensors()